In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras import layers, models
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import time
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
from keras.callbacks import EarlyStopping,ReduceLROnPlateau
from matplotlib import pyplot as plt

In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"keshrivishal21","key":"4c8bb86460f4dd386e694fd83bc26447"}'}

In [ ]:
!mkdir -p ~/.config/kaggle
!mv kaggle.json ~/.config/kaggle/
!chmod 600 ~/.config/kaggle/kaggle.json

In [ ]:
!kaggle competitions download -c state-farm-distracted-driver-detection

100% 4.00G/4.00G [00:50<00:00, 85.8MB/s]



In [ ]:
!unzip -q state-farm-distracted-driver-detection.zip -d data

In [ ]:
df = pd.read_csv("/content/data/driver_imgs_list.csv")
df.head()

,subject,classname,img
0,p002,c0,img_44733.jpg
1,p002,c0,img_72999.jpg
2,p002,c0,img_25094.jpg
3,p002,c0,img_69092.jpg
4,p002,c0,img_92629.jpg


In [ ]:
drivers = df['subject'].unique()
print("Total drivers:", len(drivers))

Total drivers: 26


In [ ]:
train_drivers, temp_drivers = train_test_split(
    drivers,
    test_size=0.4,
    random_state=42
)

val_drivers, test_drivers = train_test_split(
    temp_drivers,
    test_size=0.5,
    random_state=42
)

print("Train drivers:", len(train_drivers))
print("Val drivers:", len(val_drivers))
print("Test drivers:", len(test_drivers))

Train drivers: 15
Val drivers: 5
Test drivers: 6


In [ ]:
train_df = df[df['subject'].isin(train_drivers)]
val_df = df[df['subject'].isin(val_drivers)]
test_df = df[df['subject'].isin(test_drivers)]

print(train_df['subject'].nunique())
print(val_df['subject'].nunique())
print(test_df['subject'].nunique())

15
5
6


In [ ]:
base_path = "/content/data/imgs/train"

def add_image_path(df):
    return df.apply(
        lambda row: os.path.join(base_path, row['classname'], row['img']),
        axis=1
    )

train_paths = add_image_path(train_df)
val_paths = add_image_path(val_df)
test_paths = add_image_path(test_df)

train_labels = train_df['classname']
val_labels = val_df['classname']
test_labels = test_df['classname']

In [ ]:
le = LabelEncoder()
train_labels = le.fit_transform(train_labels)
val_labels = le.transform(val_labels)
test_labels = le.transform(test_labels)

In [ ]:
IMG_SIZE = (256,256)
BATCH_SIZE = 12
tf.random.set_seed(42)

def process_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)

    h = tf.shape(image)[0]
    w = tf.shape(image)[1]

    top = tf.cast(0.25 * tf.cast(h, tf.float32), tf.int32)
    bottom = tf.cast(0.95 * tf.cast(h, tf.float32), tf.int32)
    left = tf.cast(0.15 * tf.cast(w, tf.float32), tf.int32)
    right = tf.cast(0.85 * tf.cast(w, tf.float32), tf.int32)

    image = image[top:bottom, left:right]

    image = tf.cast(image, tf.float32)
    image = tf.image.resize(image, IMG_SIZE, method='bilinear', antialias=True)

    return image, label

def create_dataset(paths, labels, shuffle=True):
    labels = tf.convert_to_tensor(labels, dtype=tf.int32)
    labels = tf.one_hot(labels, depth=10)

    paths = paths.values

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    ds = ds.map(process_image, num_parallel_calls=tf.data.AUTOTUNE)

    if shuffle:
        ds = ds.shuffle(buffer_size=1500)

    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds


train_ds = create_dataset(train_paths, train_labels, shuffle=True)
val_ds = create_dataset(val_paths, val_labels, shuffle=False)
test_ds = create_dataset(test_paths, test_labels, shuffle=False)

In [ ]:
for path in train_paths[:5]:
    print(path, os.path.exists(path))

/content/data/imgs/train/c0/img_48693.jpg True
/content/data/imgs/train/c0/img_44903.jpg True
/content/data/imgs/train/c0/img_58514.jpg True
/content/data/imgs/train/c0/img_62307.jpg True
/content/data/imgs/train/c0/img_83984.jpg True


In [ ]:
set(train_drivers) & set(val_drivers)
set(train_drivers) & set(test_drivers)
set(val_drivers) & set(test_drivers)

set()

In [ ]:
for images, labels in train_ds.take(1):
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)

Image batch shape: (12, 256, 256, 3)
Label batch shape: (12, 10)


In [ ]:
print("Train images:", len(train_paths))
print("Val images:", len(val_paths))
print("Test images:", len(test_paths))

Train images: 13440
Val images: 3312
Test images: 5672


In [ ]:
def mobilenet_driver_model(input_shape=(256,256,3), num_classes=10):

    inputs = layers.Input(shape=input_shape)

    # Augmentation
    x = data_augmentation(inputs)

    # Preprocessing
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x)

    # Base Model
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet'
    )

    base_model.trainable = False

    x = base_model(x, training=False)

    # Classification Head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)

    return model, base_model

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=12,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,        # 🔥 smoother decay
    patience=3,        # 🔥 less aggressive
    min_lr=1e-7        # 🔥 prevent too small LR
)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),   # 🔥 reduced
    layers.RandomZoom(0.2),        # 🔥 reduced
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.15), # 🔥 slightly reduced
])

In [ ]:
model, base_model = mobilenet_driver_model()
model.summary()

/tmp/ipykernel_1096/245321981.py:12: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = tf.keras.applications.MobileNetV2(


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 8, 8, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,428,874 (9.27 MB)

 Trainable params: 168,074 (656.54 KB)

 Non-trainable params: 2,260,800 (8.62 MB)

In [ ]:

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=35,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/35
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 115s 85ms/step - accuracy: 0.2897 - loss: 2.3389 - val_accuracy: 0.3750 - val_loss: 2.0990 - learning_rate: 1.0000e-04
Epoch 2/35
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 127s 80ms/step - accuracy: 0.4859 - loss: 1.7480 - val_accuracy: 0.4016 - val_loss: 2.0400 - learning_rate: 1.0000e-04
Epoch 3/35
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 139s 76ms/step - accuracy: 0.5681 - loss: 1.5390 - val_accuracy: 0.4173 - val_loss: 2.0130 - learning_rate: 1.0000e-04
Epoch 4/35
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 143s 78ms/step - accuracy: 0.6213 - loss: 1.4078 - val_accuracy: 0.4088 - val_loss: 2.0541 - learning_rate: 1.0000e-04
Epoch 5/35
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 95s 79ms/step - accuracy: 0.6421 - loss: 1.3595 - val_accuracy: 0.4260 - val_loss: 1.9925 - learning_rate: 1.0000e-04
Epoch 6/35
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 137s 76ms/step - accuracy: 0.6798 - loss: 1.2840 - val_accuracy: 0.4290 - val_loss: 1.9866 - learning_rate: 1.0000e-04
Epoch 7/35
1120/1120 ━━━━━━━━━━━━━━

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print("Driver-wise Test Accuracy:", test_acc)

473/473 ━━━━━━━━━━━━━━━━━━━━ 32s 67ms/step - accuracy: 0.5257 - loss: 1.6465
Driver-wise Test Accuracy: 0.5257405042648315


In [ ]:
base_model.trainable = True

# Freeze most layers, train last 80
for layer in base_model.layers[:-80]:
    layer.trainable = False

In [ ]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 8, 8, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,765,024 (10.55 MB)

 Trainable params: 2,235,786 (8.53 MB)

 Non-trainable params: 193,088 (754.25 KB)

 Optimizer params: 336,150 (1.28 MB)

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint = ModelCheckpoint(
    "mobileNet_best_model.keras",
    monitor='val_loss',
    save_best_only=True
)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-6),  # lower LR
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy']
)
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=35,
    callbacks=[early_stop, reduce_lr, checkpoint]
)

Epoch 1/35
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 145s 107ms/step - accuracy: 0.4440 - loss: 1.7296 - val_accuracy: 0.4203 - val_loss: 2.0265 - learning_rate: 3.0000e-06
Epoch 2/35
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 125s 106ms/step - accuracy: 0.5603 - loss: 1.4086 - val_accuracy: 0.4402 - val_loss: 1.9031 - learning_rate: 3.0000e-06
Epoch 3/35
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 145s 109ms/step - accuracy: 0.6413 - loss: 1.2139 - val_accuracy: 0.4668 - val_loss: 1.8029 - learning_rate: 3.0000e-06
Epoch 4/35
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 142s 110ms/step - accuracy: 0.6948 - loss: 1.0835 - val_accuracy: 0.4882 - val_loss: 1.7515 - learning_rate: 3.0000e-06
Epoch 5/35
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 130s 111ms/step - accuracy: 0.7379 - loss: 0.9882 - val_accuracy: 0.4988 - val_loss: 1.7105 - learning_rate: 3.0000e-06
Epoch 6/35
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 125s 105ms/step - accuracy: 0.7673 - loss: 0.9137 - val_accuracy: 0.5236 - val_loss: 1.6640 - learning_rate: 3.0000e-06
Epoch 7/35
1120/1120 ━━━━━━━

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print("Driver-wise Test Accuracy:", test_acc)

473/473 ━━━━━━━━━━━━━━━━━━━━ 31s 66ms/step - accuracy: 0.6656 - loss: 1.2608
Driver-wise Test Accuracy: 0.6655500531196594


**Using weighted decay**

In [ ]:
model.load_weights("mobileNet_best_model.keras")

In [ ]:
from tensorflow.keras.optimizers import AdamW

model.compile(
    optimizer=AdamW(
        learning_rate=3e-6,
        weight_decay=1e-5
    ),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy']
)

In [ ]:
history_fine_2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=35,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/35
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 153s 113ms/step - accuracy: 0.9079 - loss: 0.5872 - val_accuracy: 0.5655 - val_loss: 1.5915 - learning_rate: 3.0000e-06
Epoch 2/35
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 135s 108ms/step - accuracy: 0.9140 - loss: 0.5774 - val_accuracy: 0.5719 - val_loss: 1.5682 - learning_rate: 3.0000e-06
Epoch 3/35
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 131s 112ms/step - accuracy: 0.9171 - loss: 0.5693 - val_accuracy: 0.5710 - val_loss: 1.5595 - learning_rate: 3.0000e-06
Epoch 4/35
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 137s 108ms/step - accuracy: 0.9202 - loss: 0.5607 - val_accuracy: 0.5694 - val_loss: 1.5704 - learning_rate: 3.0000e-06
Epoch 5/35
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 141s 106ms/step - accuracy: 0.9233 - loss: 0.5520 - val_accuracy: 0.5613 - val_loss: 1.6140 - learning_rate: 3.0000e-06
Epoch 6/35
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 147s 112ms/step - accuracy: 0.9272 - loss: 0.5463 - val_accuracy: 0.5682 - val_loss: 1.5851 - learning_rate: 3.0000e-06
Epoch 7/35
1120/1120 ━━━━━━━

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print("Driver-wise Test Accuracy (AdamW):", test_acc)

473/473 ━━━━━━━━━━━━━━━━━━━━ 27s 57ms/step - accuracy: 0.6747 - loss: 1.2345
Driver-wise Test Accuracy (AdamW): 0.674717903137207


In [ ]:
import time
import numpy as np

sample_batch = next(iter(test_ds))[0]

start = time.time()
pred = model.predict(sample_batch)
end = time.time()

time_per_image = (end - start) / len(sample_batch)
print("Time per image (seconds):", time_per_image)
time_per_image = time_per_image * 1000
print("Time per image (milliseconds):", time_per_image)

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Time per image (seconds): 0.15963417291641235
Time per image (milliseconds): 159.63417291641235
